In [1]:
import networkx as nx
import pandas as pd
import numpy as np
from rdflib import URIRef
import pickle

## Process NP-KG and reference data for embeddings pipeline

In [2]:
FILE_PATH = '/home/sanya/npkg-workspace/reference-data/mapped_processed_data/combined_interactions_nodrugclass_dedup_20240830.tsv'
DIR_OUT = 'data/reference-data/'

In [3]:
refdf = pd.read_csv(FILE_PATH, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1462 non-null   object
 1   np_id              1462 non-null   object
 2   np                 1462 non-null   object
 3   drug_id            1462 non-null   object
 4   drug               1462 non-null   object
 5   interaction_type   1461 non-null   object
 6   source_data        1462 non-null   object
 7   protein_target     655 non-null    object
 8   adr_label          217 non-null    object
 9   adr_uri            217 non-null    object
 10  llm-interact-type  1049 non-null   object
 11  mech-type          1049 non-null   object
 12  adr                40 non-null     object
 13  related_np         1462 non-null   object
 14  related_np_id      1462 non-null   object
dtypes: object(15)
memory usage: 171.5+ KB


In [4]:
refdf.loc[refdf['interaction_type'].isna()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,llm-interact-type,mech-type,adr,related_np,related_np_id
119,stockley_163,http://napdi.org/napdi_srs_imports:glycine_max,Soy,http://purl.obolibrary.org/obo/CHEBI_45863,Paclitaxel,NaN,Stockleys,NaN,NaN,NaN,None,pharmacokinetic,NaN,Soy,http://napdi.org/napdi_srs_imports:glycine_max


In [5]:
refdf.at[119, 'interaction_type'] = 'negative'

In [6]:
refdf['interaction_type'] = refdf['interaction_type'].str.lower()
refdf['interaction_type'].value_counts()

positive                                                                                                                                                                                                                                                                         792
positive|positive                                                                                                                                                                                                                                                                249
negative                                                                                                                                                                                                                                                                         119
positive|positive|positive|positive                                                                                                                                      

In [7]:
for i in range(len(refdf.index)):
    interaction_type_list = refdf.at[i, 'interaction_type'].split('|')
    interaction_type = list(set(interaction_type_list))
    if 'positive' in interaction_type and 'negative' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'both'
    elif 'positive' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'positive'
    elif 'negative' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'negative'

In [8]:
#positive    1208
#negative     161
#both          86
refdf['interaction_type'].value_counts()

positive    1215
negative     161
both          86
Name: interaction_type, dtype: int64

In [9]:
##import graph
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
NodeLabelsFile = KG_PATH + 'nodeLabels_v3.0.0.pickle'
#KG_NAME_MERGED = 'NP-KG_v2.0.0.gpickle'
KG_NAME_MERGED_WEIGHTED = 'NP-KG_BETWEENNESS-WEIGHTED-PK-HASHED_v3.0.0.gpickle'

In [10]:
with open(KG_PATH+KG_NAME_MERGED_WEIGHTED, 'rb') as f:
    npkg = pickle.load(f)

In [11]:
with open(NodeLabelsFile, 'rb') as filep:
    nodeLabels = pickle.load(filep)

In [19]:
def get_graph_stats(kg):
    nodes = nx.number_of_nodes(kg)
    edges = nx.number_of_edges(kg)
    self_loops = nx.number_of_selfloops(kg)

    print('There are {} nodes, {} edges, and {} self-loop(s)'.format(nodes, edges, self_loops))
    # get degree information
    avg_degree = float(edges) / nodes
    print('The Average Degree is {}'.format(avg_degree))
    
    print('Nodes with highest degree:')
    n_deg = sorted([(str(x[0]), x[1]) for x in  kg.degree], key=lambda x: x[1], reverse=1)[:6]

    for x in n_deg:
        print('Label: {}'.format(nodeLabels[x[0]]))
        print('{} (degree={})'.format(x[0], x[1]))
    # get network density
    density = nx.density(kg)

    print('The density of the graph is: {}'.format(density))

In [20]:
get_graph_stats(npkg)

There are 1089139 nodes, 7786937 edges, and 453 self-loop(s)
The Average Degree is 7.149626448047495
Nodes with highest degree:
Label: SNV
http://purl.obolibrary.org/obo/SO_0001483 (degree=214261)
Label: protein_coding_gene
http://purl.obolibrary.org/obo/SO_0001217 (degree=189350)
Label: transcript
http://purl.obolibrary.org/obo/SO_0000673 (degree=178309)
Label: Homo sapiens
http://purl.obolibrary.org/obo/NCBITaxon_9606 (degree=91837)
Label: protein
http://purl.obolibrary.org/obo/PR_000000001 (degree=56320)
Label: Mus musculus
http://purl.obolibrary.org/obo/NCBITaxon_10090 (degree=53597)
The density of the graph is: 6.5644816800511e-06


In [12]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,llm-interact-type,mech-type,adr,related_np,related_np_id
0,stockley_0,http://napdi.org/napdi_srs_imports:withania_so...,Ashwagandha,http://purl.obolibrary.org/obo/CHEBI_4551,Digoxin,positive,Stockleys,NaN,NaN,NaN,None,pharmacodynamic,NaN,Ashwagandha,http://napdi.org/napdi_srs_imports:withania_so...
1,stockley_3|napdi_69|napdi_70|napdi_71|ddid_366,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine|berberine|berberine|berberine|Berberine,http://purl.obolibrary.org/obo/CHEBI_4031,Ciclosporin|cyclosporin a|cyclosporin a|cyclos...,both,Stockleys|NaPDI|NaPDI|NaPDI|DDID,P-glycoprotein|CYP3A4|P-gp (ABCB1)|CYP3A4|P-gp...,NaN,NaN,inhibition|inhibition,pharmacokinetic|pharmacokinetic,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118
2,stockley_4,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine,http://purl.obolibrary.org/obo/CHEBI_16794,Hyoscine (Scopolamine),positive,Stockleys,NaN,NaN,NaN,None,None,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118
3,stockley_5|napdi_73,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_45863,Paclitaxel|paclitaxel,both,Stockleys|NaPDI,CYP2C8,NaN,NaN,None,None,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118
4,stockley_7,http://napdi.org/napdi_srs_imports:cinnamomum_...,Cinnamon,http://purl.obolibrary.org/obo/CHEBI_3387,Carbamazepine,negative,Stockleys,NaN,NaN,NaN,None,pharmacokinetic,NaN,Cinnamon,http://napdi.org/napdi_srs_imports:cinnamomum_...


In [13]:
refdf['npkg_edge'] = pd.NA
refdf['npkg_edge_label'] = pd.NA

In [14]:
for i in range(len(refdf.index)):
    np_id = refdf.loc[i, 'np_id']
    drug_id = refdf.loc[i, 'drug_id']
    source = URIRef(np_id)
    target = URIRef(drug_id)
    if source in npkg and target in npkg:
        if npkg.has_edge(source, target):
            edge_data = npkg.get_edge_data(source, target)
            refdf.at[i, 'npkg_edge'] = edge_data['key']
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1462 non-null   object
 1   np_id              1462 non-null   object
 2   np                 1462 non-null   object
 3   drug_id            1462 non-null   object
 4   drug               1462 non-null   object
 5   interaction_type   1462 non-null   object
 6   source_data        1462 non-null   object
 7   protein_target     655 non-null    object
 8   adr_label          217 non-null    object
 9   adr_uri            217 non-null    object
 10  llm-interact-type  1049 non-null   object
 11  mech-type          1049 non-null   object
 12  adr                40 non-null     object
 13  related_np         1462 non-null   object
 14  related_np_id      1462 non-null   object
 15  npkg_edge          84 non-null     object
 16  npkg_edge_label    0 non-null      object


In [15]:
refdf.loc[refdf['npkg_edge'].notnull()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,llm-interact-type,mech-type,adr,related_np,related_np_id,npkg_edge,npkg_edge_label
1,stockley_3|napdi_69|napdi_70|napdi_71|ddid_366,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine|berberine|berberine|berberine|Berberine,http://purl.obolibrary.org/obo/CHEBI_4031,Ciclosporin|cyclosporin a|cyclosporin a|cyclos...,both,Stockleys|NaPDI|NaPDI|NaPDI|DDID,P-glycoprotein|CYP3A4|P-gp (ABCB1)|CYP3A4|P-gp...,NaN,NaN,inhibition|inhibition,pharmacokinetic|pharmacokinetic,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118,http://purl.obolibrary.org/obo/RO_0002213|http...,<NA>
67,stockley_92|napdi_229,http://napdi.org/napdi_srs_imports:camellia_si...,Green Tea|green tea leaf,http://purl.obolibrary.org/obo/CHEBI_27732,Caffeine|caffeine,negative,Stockleys|NaPDI,CYP1A2,NaN,NaN,None,pharmacokinetic,NaN,Green Tea,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002180|http...,<NA>
73,stockley_99|natmed_6|fideo_310,http://napdi.org/napdi_srs_imports:camellia_si...,Green Tea|Green tea| tea food product,http://purl.obolibrary.org/obo/CHEBI_10033,Warfarin and related drugs|WARFARIN (Coumadin)...,both,Stockleys|NatMedPro|FIDEO,NaN,Decreased prothrombin time|Abnormal bleeding,http://purl.obolibrary.org/obo/HP_0032198|http...,None|None,pharmacodynamic|pharmacodynamic,NaN,Green Tea,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002449,<NA>
97,stockley_135,http://napdi.org/napdi_srs_imports:panax_ginseng,Panax Ginseng,http://purl.obolibrary.org/obo/CHEBI_3655,Chlorzoxazone,negative,Stockleys,NaN,NaN,NaN,None,pharmacokinetic,NaN,Panax Ginseng,http://napdi.org/napdi_srs_imports:panax_ginseng,http://purl.obolibrary.org/obo/RO_0002434,<NA>
100,stockley_138|natmed_296,http://napdi.org/napdi_srs_imports:panax_ginseng,Panax Ginseng|Panax ginseng,http://purl.obolibrary.org/obo/CHEBI_5050,Fexofenadine|FEXOFENADINE (Allegra),both,Stockleys|NatMedPro,P-glycoprotein,NaN,NaN,None|induction,None|pharmacokinetic,NaN,Panax Ginseng,http://napdi.org/napdi_srs_imports:panax_ginseng,http://purl.obolibrary.org/obo/RO_0002434,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359,ddid_1448|ddid_1449|ddid_1452,http://napdi.org/napdi_srs_imports:allium_sativum,Garlic|Garlic|Garlic,http://purl.obolibrary.org/obo/CHEBI_45409,Ritonavir|Ritonavir|Ritonavir,both,DDID|DDID|DDID,P-gp|P-gp,NaN,NaN,None|inhibition|None,pharmacokinetic|pharmacokinetic|pharmacokinetic,NaN,Garlic,http://napdi.org/napdi_srs_imports:allium_sativum,http://purl.obolibrary.org/obo/RO_0002434|http...,<NA>
1385,ddid_1482|ddid_1483|ddid_1484|ddid_1485|ddid_1497,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-Epigallocatechin-3-Gallate|(-)-Epigallocat...,http://purl.obolibrary.org/obo/CHEBI_38545,Rosuvastatin|Rosuvastatin|Rosuvastatin|Rosuvas...,positive,DDID|DDID|DDID|DDID|DDID,OATP1B1|OATP1B1|OATP1B1,NaN,NaN,None|inhibition|inhibition|inhibition|None,pharmacokinetic|pharmacokinetic|pharmacokineti...,NaN,Camellia sinensis [Syn. Thea sinensis],http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002434|http...,<NA>
1401,ddid_1577,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-Epigallocatechin-3-Gallate,http://purl.obolibrary.org/obo/CHEBI_38940,Sunitinib,positive,DDID,NaN,NaN,NaN,None,pharmacokinetic,NaN,Camellia sinensis [Syn. Thea sinensis],http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002434|http...,<NA>
1405,ddid_1593,http://purl.obolibrary.org/obo/CHEBI_125451,Silymarin,http://purl.obolibrary.org/obo/CHEBI_135533,Talinolol,positive,DDID,NaN,NaN,NaN,None,pharmacokinetic,NaN,Silybum Marianum,http://napdi.org/napdi_srs_imports:silybum_mar...,http://purl.obolibrary.org/obo/RO_0002213,<NA>


In [16]:
for i in range(len(refdf.index)):
    npkg_edge = refdf.loc[i, 'npkg_edge']
    if not pd.isna(npkg_edge):
        edge_list = npkg_edge.split('|')
        if len(edge_list) == 1:
            refdf.at[i, 'npkg_edge_label'] = nodeLabels.get(edge_list[0], '')
        else:
            edge_label_list = [nodeLabels.get(x, '') for x in edge_list]
            refdf.at[i, 'npkg_edge_label'] = '|'.join(edge_label_list)
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1462 non-null   object
 1   np_id              1462 non-null   object
 2   np                 1462 non-null   object
 3   drug_id            1462 non-null   object
 4   drug               1462 non-null   object
 5   interaction_type   1462 non-null   object
 6   source_data        1462 non-null   object
 7   protein_target     655 non-null    object
 8   adr_label          217 non-null    object
 9   adr_uri            217 non-null    object
 10  llm-interact-type  1049 non-null   object
 11  mech-type          1049 non-null   object
 12  adr                40 non-null     object
 13  related_np         1462 non-null   object
 14  related_np_id      1462 non-null   object
 15  npkg_edge          84 non-null     object
 16  npkg_edge_label    84 non-null     object


In [19]:
refdf.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240829_with_npkg_edges.tsv', sep='\t', index=False)

In [20]:
refdf_nokg = refdf.loc[refdf['npkg_edge'].isna()]
refdf_nokg = refdf_nokg.drop(columns=['npkg_edge', 'npkg_edge_label'])
refdf_nokg = refdf_nokg.reset_index(drop=True)
refdf_nokg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1378 entries, 0 to 1377
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1378 non-null   object
 1   np_id              1378 non-null   object
 2   np                 1378 non-null   object
 3   drug_id            1378 non-null   object
 4   drug               1378 non-null   object
 5   interaction_type   1378 non-null   object
 6   source_data        1378 non-null   object
 7   protein_target     598 non-null    object
 8   adr_label          208 non-null    object
 9   adr_uri            208 non-null    object
 10  llm-interact-type  992 non-null    object
 11  mech-type          992 non-null    object
 12  adr                40 non-null     object
 13  related_np         1378 non-null   object
 14  related_np_id      1378 non-null   object
dtypes: object(15)
memory usage: 161.6+ KB


In [21]:
refdf_nokg.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240829_without_npkg_edges.tsv', sep='\t', index=False)

### Load data and convert to CURIEs -- save only edges for prediction

In [22]:
refdf = pd.read_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240829_without_npkg_edges.tsv', sep='\t')

In [23]:
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1378 entries, 0 to 1377
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1378 non-null   object
 1   np_id              1378 non-null   object
 2   np                 1378 non-null   object
 3   drug_id            1378 non-null   object
 4   drug               1378 non-null   object
 5   interaction_type   1378 non-null   object
 6   source_data        1378 non-null   object
 7   protein_target     598 non-null    object
 8   adr_label          208 non-null    object
 9   adr_uri            208 non-null    object
 10  llm-interact-type  992 non-null    object
 11  mech-type          992 non-null    object
 12  adr                40 non-null     object
 13  related_np         1378 non-null   object
 14  related_np_id      1378 non-null   object
dtypes: object(15)
memory usage: 161.6+ KB


In [24]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,llm-interact-type,mech-type,adr,related_np,related_np_id
0,stockley_0,http://napdi.org/napdi_srs_imports:withania_so...,Ashwagandha,http://purl.obolibrary.org/obo/CHEBI_4551,Digoxin,positive,Stockleys,NaN,NaN,NaN,None,pharmacodynamic,NaN,Ashwagandha,http://napdi.org/napdi_srs_imports:withania_so...
1,stockley_4,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine,http://purl.obolibrary.org/obo/CHEBI_16794,Hyoscine (Scopolamine),positive,Stockleys,NaN,NaN,NaN,None,None,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118
2,stockley_5|napdi_73,http://purl.obolibrary.org/obo/CHEBI_16118,Berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_45863,Paclitaxel|paclitaxel,both,Stockleys|NaPDI,CYP2C8,NaN,NaN,None,None,NaN,Berberine,http://purl.obolibrary.org/obo/CHEBI_16118
3,stockley_7,http://napdi.org/napdi_srs_imports:cinnamomum_...,Cinnamon,http://purl.obolibrary.org/obo/CHEBI_3387,Carbamazepine,negative,Stockleys,NaN,NaN,NaN,None,pharmacokinetic,NaN,Cinnamon,http://napdi.org/napdi_srs_imports:cinnamomum_...
4,stockley_8,http://napdi.org/napdi_srs_imports:cinnamomum_...,Cinnamon,http://purl.obolibrary.org/obo/CHEBI_7731,Ofloxacin,negative,Stockleys,NaN,NaN,NaN,None,pharmacokinetic,NaN,Cinnamon,http://napdi.org/napdi_srs_imports:cinnamomum_...


In [25]:
uri_to_curie_map_file = '/home/sanya/np-kg/resources/knowledge_graphs/uri_to_curie_map.tsv'
##load as dictionary
uri_to_curie_map = {}
with open(uri_to_curie_map_file, 'r') as f:
    for line in f:
        uri, curie = line.strip().split('\t')
        uri_to_curie_map[uri] = curie
len(uri_to_curie_map)

1089472

In [26]:
for key in uri_to_curie_map:
    print(key, uri_to_curie_map[key])
    break

http://purl.obolibrary.org/obo/CLO_0035452 clo:0035452


In [27]:
##add missing curies
uri_to_curie_map['http://purl.obolibrary.org/obo/CHEBI:9654'] = 'chebi:9654'

In [28]:
edge_df = pd.DataFrame(columns=['source', 'target', 'interaction_type'])
for i in range(len(refdf.index)):
    np_id = refdf.at[i, 'np_id']
    drug_id = refdf.at[i, 'drug_id']
    interaction_type = refdf.at[i, 'interaction_type']
    if np_id in uri_to_curie_map and drug_id in uri_to_curie_map:
        source = uri_to_curie_map[np_id]
        target = uri_to_curie_map[drug_id]
        tempdf = pd.DataFrame({'source': [source], 'target': [target], 'interaction_type': [interaction_type]})
        edge_df = pd.concat([edge_df, tempdf], ignore_index=True)
    else:
        print('URI not found in map: {} {}'.format(np_id, drug_id))
edge_df.info()

URI not found in map: http://napdi.org/napdi_srs_imports:serenoa_repens http://purl.obolibrary.org/obo/CHEBI_4475
URI not found in map: http://napdi.org/napdi_srs_imports:valeriana_officinalis http://purl.obolibrary.org/obo/CHEBI_4476
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229213
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_195558
URI not found in map: http://purl.obolibrary.org/obo/CHEBI_207762 http://purl.obolibrary.org/obo/CHEBI_8772
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_230905
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_94311
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229526
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi 

In [30]:
edge_df.head()

,source,target,interaction_type
0,napdi_srs_imports:withania_somnifera,chebi:4551,positive
1,chebi:16118,chebi:16794,positive
2,chebi:16118,chebi:45863,both
3,napdi_srs_imports:cinnamomum_verum,chebi:3387,negative
4,napdi_srs_imports:cinnamomum_verum,chebi:7731,negative


In [31]:
edge_df.to_csv(DIR_OUT+'reference_data_curie_edges_20240829.tsv', sep='\t', index=False)

In [21]:
####check if new nodes exist in KG
import pickle
import networkx as nx
from rdflib import URIRef

In [19]:
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
KG_NAME = 'NP-KG_BETWEENNESS-WEIGHTED-PK-HASHED_v3.0.0.gpickle'

In [20]:
with open(KG_PATH+KG_NAME, 'rb') as f:
    npkg = pickle.load(f)

In [23]:
nodelist = ['http://purl.obolibrary.org/obo/CHEBI_4475','http://purl.obolibrary.org/obo/CHEBI_4476','http://purl.obolibrary.org/obo/CHEBI_229213',
'http://purl.obolibrary.org/obo/CHEBI_207762','http://purl.obolibrary.org/obo/CHEBI_230905','http://purl.obolibrary.org/obo/CHEBI_94311',
'http://purl.obolibrary.org/obo/CHEBI_229526','http://purl.obolibrary.org/obo/CHEBI_9654']

for node in nodelist:
    if URIRef(node) not in npkg:
        print(node)

http://purl.obolibrary.org/obo/CHEBI_4475
http://purl.obolibrary.org/obo/CHEBI_4476
http://purl.obolibrary.org/obo/CHEBI_229213
http://purl.obolibrary.org/obo/CHEBI_207762
http://purl.obolibrary.org/obo/CHEBI_230905
http://purl.obolibrary.org/obo/CHEBI_94311
http://purl.obolibrary.org/obo/CHEBI_229526
